Build Data

In [112]:
from core.data import build_global_data

global_data = build_global_data()

Build Static Master Graph

In [113]:
from core.graph import build_master_graph

G_master = build_master_graph(global_data)

Setup Player Team Masteries (Runs once at start of game)

In [114]:
from core.hero_mastery import create_empty_masteries, set_mastery
t1_masteries = create_empty_masteries(G_master)

# Set the masteries of the heroes you have to 1
# Top 
set_mastery(t1_masteries, "Frank", "Top", 2)
set_mastery(t1_masteries, "Hakuna", "Top", 1)
set_mastery(t1_masteries, "Justice", "Top", 1)
set_mastery(t1_masteries, "Tiger Boy", "Top", 1)

# Jungle
set_mastery(t1_masteries, "Kamaitachi", "Jungler", 2)
set_mastery(t1_masteries, "Hakuna", "Jungler", 1)
set_mastery(t1_masteries, "Zealot", "Jungler", 1)
set_mastery(t1_masteries, "Xiangxi Ke", "Jungler", 1)

# Mid
set_mastery(t1_masteries, "Aurelio", "Mid", 1)
set_mastery(t1_masteries, "Xiangxi Ke", "Mid", 1)
set_mastery(t1_masteries, "Fatty White", "Mid", 1)
set_mastery(t1_masteries, "Elemi", "Mid", 1)
set_mastery(t1_masteries, "Wolfgang", "Mid", 1)

# Bot
set_mastery(t1_masteries, "Gang", "Bot", 2)
set_mastery(t1_masteries, "Niels", "Bot", 1)
set_mastery(t1_masteries, "Bariel", "Bot", 1)
set_mastery(t1_masteries, "Omaha", "Bot", 1)
set_mastery(t1_masteries, "Shougong Lei", "Bot", 1)
set_mastery(t1_masteries, "Elemi", "Bot", 1)

# Support
set_mastery(t1_masteries, "Paisai", "Support", 1)
set_mastery(t1_masteries, "Palulu", "Support", 1)
set_mastery(t1_masteries, "Dylan", "Support", 1)
set_mastery(t1_masteries, "Fatty White", "Support", 1)
set_mastery(t1_masteries, "Peiniang Zhu", "Support", 2)
set_mastery(t1_masteries, "Tiger Boy", "Support", 1)

Start of Draft (Runs before each match)

Setup Availabiltiies and Opponent Signitures

In [115]:
from core.draft import build_position_availability
from core.hero_mastery import set_mastery

# We know what we can pick, because we know our masteries
t1_available = build_position_availability(t1_masteries)

# Build out masteries to the degree you want to
t2_masteries = create_empty_masteries(G_master)

# Top
set_mastery(t2_masteries, "Miki", "Top", 1)
set_mastery(t2_masteries, "Aurelio", "Top", 1)
set_mastery(t2_masteries, "Wolfgang", "Top", 1)

# Jungler
set_mastery(t2_masteries, "Kamaitachi", "Jungler", 1)
set_mastery(t2_masteries, "Aurelio", "Jungler", 1)
set_mastery(t2_masteries, "Hakuna", "Jungler", 1)

# Mid
set_mastery(t2_masteries, "Miki", "Mid", 1)
set_mastery(t2_masteries, "Dylan", "Mid", 1)
set_mastery(t2_masteries, "Elemi", "Mid", 1)

# Bot
set_mastery(t2_masteries, "Deep Space", "Bot", 1)
set_mastery(t2_masteries, "Elemi", "Bot", 1)
set_mastery(t2_masteries, "Niels", "Bot", 1)

# Support
set_mastery(t2_masteries, "Peiniang Zhu", "Support", 1)
set_mastery(t2_masteries, "Fatty White", "Support", 1)
set_mastery(t2_masteries, "Dylan", "Support", 1)

# One the user presses confirm (either before the draft, or after they've finished in the draft, we confirm)
t2_available = build_position_availability(t2_masteries)

Apply Mastery and Signitures to Graph

In [116]:
from core.graph import confirm_hero_masteries, _clear_hero_masteries

_clear_hero_masteries(G_master)
confirm_hero_masteries(G_master, t1_masteries, t2_masteries)

Begin Draft - Recommendations

In [117]:
from core.draft import build_draft_state
draft_state = build_draft_state()
draft_state.t1_available = t1_available
draft_state.t1_picked = {}
draft_state.t2_available = t2_available
draft_state.t2_picked = {}
draft_state.banned = set()

Recommend Pick

In [ ]:
from core.draft import recommend_pick

# Recommend a Pick
best, score, explanation, flag, all_results = recommend_pick(
    G_master, 
    "t1",
    "Jungler", 
    draft_state,
    global_data
)

# Pass outputs here

print(f"Recommended: {best} ({score})")
for reason in explanation:
    output = f"- {reason}:"
    for hero in explanation[reason]:
        output += f" {hero},"
    print(output)
if flag:
    print(f"WARNING: {flag}")
print("")


Recommended: Hakuna (9)
- position_tier: A,
- position_mastery: 1,



Recommend Ban

In [119]:
from core.draft import recommend_pick

# Recommend a Pick
best, score, explanation, flag, all_results = recommend_pick(
    G_master, 
    "t2",
    "Jungler", 
    draft_state,
    global_data
)

print(f"Recommended: {best} ({score})")
for reason in explanation:
    output = f"- {reason}:"
    for hero in explanation[reason]:
        output += f" {hero},"
    print(output)
if flag:
    print(f"WARNING: {flag}")
print("")


Recommended: Aurelio (11.5)
- synergy_possible: Dylan,
- position_tier: S,
- position_mastery: 1,



Pick Heros

In [118]:
from core.draft import pick_hero

#pick_hero(G_master, "Aurelio", "t1", draft_state)
pick_hero(G_master, "Kamaitachi", "t2", draft_state)

In [52]:
from core.draft import ban_hero

ban_hero("Tiger Boy", draft_state)

In [71]:
from core.draft import see_current_draft

see_current_draft("t1", draft_state)

{'Peiniang Zhu': {'Support'},
 'Frank': {'Top'},
 'Zealot': {'Jungler'},
 'Gang': {'Bot'},
 'Aurelio': {'Mid'}}

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

from core.data import read_hero_info, read_build_type_itemisation

@tool 
def recommend_pick_for_position(position: str):
    """Uses the master graph to find the current highest scoring hero in this position.

    Args:
        position (str): the position to check for: Top, Jungler, Mid, Bot, Support

    Returns:
        dict: Draft recommendation containing:
                recommended_hero: The highest-scoring hero for the requested lane.
                requested_lane: The position being filled.
                score: The recommended hero's draft score.
                explanation: Factors that contributed to the score.
                better_position: Another lane where the recommended hero scores
                    higher, or None.
                candidates: Other potential picks ordered from highest to lowest
                    score, including the recommended hero as the first entry.
    """
    # Recommend a Pick
    res = recommend_pick(
        G_master, 
        "t1",
        position, 
        draft_state,
        global_data
    )

    return res


@tool
def get_hero_info(hero_name: str) -> str:
    """Get markdown notes on a specific hero by name.

    Args:
        hero_name (str): the name of the hero you want to look up
    """
    return read_hero_info(hero_name)


@tool
def get_build_type_itemisation(role: str) -> str:
    """Get markdown itemisation notes for a role or item archetype.

    Args:
        role (str): the role or item archetype to look up
    """
    return read_build_type_itemisation(role)


agent = create_agent(
    model="ollama:qwen3.5",
    tools=[get_hero_info, get_build_type_itemisation],
    system_prompt=(
        "You are a MOBA team coach. Questions given to you will be about "
        "strategy in drafts and hero playstyles."
    ),
)

In [5]:
data = build_global_data()

def get_position_best_heros(position_name: str, tier: int = 1) -> str:
    """Get the position suitabiliy for a hero for the given tier.  
    Tier 1 is the bset position, 2 is the second best position, and so on.
    If a position does not appear in the results returned, then we cannot play the hero in that position.

    Args:
        hero_name (str): the name of the hero you want to check
        tier (int, optional): the position tier you want to check. Defaults to 1.

    Returns:
        dict[str, str|list[str]]: A lookup for the heroes that score in the requested tier for the requestion position.
    """
    hero_tiers = data.hero_tiers
    heroes = {
        hero: positions[position_name]
        for hero, positions in hero_tiers.items()
        if position_name in positions
    }
    matched_heroes = []
    for hero, score in heroes.items():
        if score == tier:
            matched_heroes.append(hero)
    output = {
        "position_name": position_name,
        "heroes": matched_heroes
    }
    return output

def get_hero_best_positions(hero_name: str) -> str:
    """Get the position suitabiliy for a hero for the given tier.  
    Tier 1 is the bset position, 2 is the second best position, and so on.
    If a position does not appear in the results returned, then we cannot play the hero in that position.

    Args:
        hero_name (str): the name of the hero you want to check

    Returns:
        dict[str, str|dict[str, str]]: A lookup for each position and it's tier.
    """
    tiers = data.hero_tiers.get(hero_name, {})
    output = {
        "hero_name": hero_name,
        "positions": tiers
    }
    return output

get_position_best_heros("Jungler", 5)
#get_hero_best_positions("Deep Space")

{'position_name': 'Jungler', 'heroes': ['Aurelio', 'Kamaitachi', 'Zealot']}

In [13]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Literal

from core.data import (
    read_attribute_info,
    read_build_type_itemisation,
    read_glossary_definition,
    read_hero_info,
    read_team_comp_info,
    build_global_data
)
from pathlib import Path
SYSTEM_PROMPT_FILE = Path("system_prompt.md")
SYSTEM_PROMPT = SYSTEM_PROMPT_FILE.read_text(encoding="utf-8")
data = build_global_data()


@tool 
def get_hero_best_positions(hero_name: str) -> str:
    """Return the position suitabiliy for the requested hero for the requested tier.  

    Args:
        hero_name (str): the name of the hero you want to check

    Returns:
        dict[str, str|dict[str, str]]: A lookup for each position and it's tier.
    """
    tiers = data.hero_tiers.get(hero_name, {})
    output = {
        "hero_name": hero_name,
        "positions": tiers
    }
    return output

@tool 
def get_position_best_heroes(
    position_name: Literal["Top", "Jungler", "Mid", "Bot", "Support"], 
    tier: Literal[1, 2, 3, 4, 5] = 5) -> str:
    """Return heroes whose tier for the given position matches the requested tier.

    Args:
        position_name (str): the name of the position you want to check.
        tier (int, optional): the position tier you want to check. Defaults to 5.

    Returns:
        dict[str, str|list[str]]: A lookup for the heroes that score in the requested tier for the requestion position.
    """
    hero_tiers = data.hero_tiers
    heroes = {
        hero: positions[position_name]
        for hero, positions in hero_tiers.items()
        if position_name in positions
    }
    matched_heroes = []
    for hero, score in heroes.items():
        if score == tier:
            matched_heroes.append(hero)

    output = {
        "position_name": position_name,
        "heroes": matched_heroes
    }
    return output


@tool
def get_team_comp_info(comp_name: str) -> str:
    """Get notes on a team comp.

    Args:
        comp_name (str): The name of the team comp.

    Returns:
        str: information regarding the team comp.
    """
    return read_team_comp_info(comp_name)


@tool
def get_attribute_info(category: str, instance: str) -> str:
    """Get notes on a category of hero attributes.

    Args:
        category (str): The category (hero class, attack type, damage type)
        instance (str): The attribute name, for example (gladiator, melee, magical)

    Returns:
        str: information regarding this attribute
    """
    return read_attribute_info(category, instance)


@tool
def get_hero_info(hero_name: str) -> str:
    """Get markdown notes on a specific hero by name.

    Args:
        hero_name (str): the name of the hero you want to look up

    Returns:
        str: information regarding this hero
    """
    return read_hero_info(hero_name)


@tool
def get_build_type_itemisation(build: Literal[
    "adc",
    "aoe",
    "assassin",
    "bruiser",
    "buffer",
    "mage",
    "multi hitter",
    "one slap clap",
    "pure tank",
    "seat warmer",
    "special case",
    "tank one damage item",
]) -> str:
    """Get markdown itemisation notes for a build type.

    Args:
        role (str): the role to look up

    Returns:
        str: information on the 
    """
    return read_build_type_itemisation(build)


@tool
def lookup_glossary(term: str) -> str:
    """Define a game term or abbreviation from the glossary.

    Args:
        term (str): the term or abbreviation to define.

    Returns:
        str: the definition of the term or abbreviation.
    """
    return read_glossary_definition(term)



agent = create_agent(
    model="ollama:qwen3.5",
    tools=[
        get_hero_info, 
        get_build_type_itemisation, 
        lookup_glossary, 
        get_attribute_info, 
        get_team_comp_info,
        get_hero_best_positions,
        get_position_best_heroes
    ],
    system_prompt=SYSTEM_PROMPT
)

PROMPT = "what lane should Bajie be played in?"

stream = agent.stream_events(
    {
        "messages": [
            {"role": "user", "content": PROMPT}
        ]
    },
    version="v3"
)
for kind, item in stream.interleave("messages", "tool_calls"):
    print(kind)
    print(item)
    if kind == "messages":
        pass
        #print(item.text)
        #for token in item.text:
        #    print(token, end="", flush=True)
    if kind == "tool_calls":
        pass
        #print(item.tool_call_id)
        #print(item.input)
        #print(item.output)
        #print(f"\nTool call: {item.tool_name}({item.input})")
        #print("\n")

print(stream.output)


messages
tool_calls
ToolCallStream(tool_call_id='0e3c8eff-b480-4d1b-ae50-36804b9b1e02', tool_name='get_hero_best_positions', status=running)
messages
{'messages': [HumanMessage(content='what lane should Bajie be played in?', additional_kwargs={}, response_metadata={}, id='bb0a9e89-73d6-40a3-9d46-d14e42c7863a'), AIMessage(content=[{'type': 'tool_call', 'id': '0e3c8eff-b480-4d1b-ae50-36804b9b1e02', 'name': 'get_hero_best_positions', 'args': {'hero_name': 'Bajie'}}], additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-07-05T04:03:37.5765628Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2175554300, 'load_duration': 274370800, 'prompt_eval_count': 1700, 'prompt_eval_duration': 245773000, 'eval_count': 122, 'eval_duration': 1639283000, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama', 'output_version': 'v1'}, id='lc_run--019f3072-2767-7b60-b443-8e42afcfe5e7', tool_calls=[{'name': 'get_hero_best_positions', 'args': {'hero_name': 'B

In [5]:
from langchain_ollama import ChatOllama
agent = create_agent(
    model=ChatOllama(model="gemma4:12b"),
    tools=[
        get_hero_info, 
        get_build_type_itemisation, 
        lookup_glossary, 
        get_attribute_info, 
        get_team_comp_info,
        get_hero_best_positions,
        get_position_best_heroes
    ],
    system_prompt=SYSTEM_PROMPT
)

PROMPT = "what lane should BaJie be played in?"

res = agent.invoke({
    "messages": [
        {"role": "user", "content": PROMPT}
    ]
})


{'messages': [HumanMessage(content='what lane should BaJie be played in?', additional_kwargs={}, response_metadata={}, id='7846a3f6-f037-4d4a-ba71-a17a7f5f70f5'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:12b', 'created_at': '2026-07-05T03:22:41.2426955Z', 'done': True, 'done_reason': 'stop', 'total_duration': 16553181100, 'load_duration': 13882712300, 'prompt_eval_count': 1405, 'prompt_eval_duration': 962341000, 'eval_count': 82, 'eval_duration': 1692936000, 'logprobs': None, 'model_name': 'gemma4:12b', 'model_provider': 'ollama'}, id='lc_run--019f304c-7430-7b41-ac58-dc19b3f5cbb1-0', tool_calls=[{'name': 'get_hero_best_positions', 'args': {'hero_name': 'BaJie'}, 'id': 'cd69ebe4-a1f6-4995-8af1-853c93d41c94', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1405, 'output_tokens': 82, 'total_tokens': 1487}), ToolMessage(content='{"hero_name": "BaJie", "positions": {"Bot": 5}}', name='get_hero_best_positions', id='1f0deaac

In [7]:
print(res["messages"][1])

content='' additional_kwargs={} response_metadata={'model': 'gemma4:12b', 'created_at': '2026-07-05T03:22:41.2426955Z', 'done': True, 'done_reason': 'stop', 'total_duration': 16553181100, 'load_duration': 13882712300, 'prompt_eval_count': 1405, 'prompt_eval_duration': 962341000, 'eval_count': 82, 'eval_duration': 1692936000, 'logprobs': None, 'model_name': 'gemma4:12b', 'model_provider': 'ollama'} id='lc_run--019f304c-7430-7b41-ac58-dc19b3f5cbb1-0' tool_calls=[{'name': 'get_hero_best_positions', 'args': {'hero_name': 'BaJie'}, 'id': 'cd69ebe4-a1f6-4995-8af1-853c93d41c94', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 1405, 'output_tokens': 82, 'total_tokens': 1487}


In [17]:
for message in res["messages"]:
    print(type(message))
    print(message)
    print("")

<class 'langchain_core.messages.human.HumanMessage'>
content='what lane should BaJie be played in?' additional_kwargs={} response_metadata={} id='fe84cb6e-38ed-4cfd-bad7-b82d6527c1d9'

<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={} response_metadata={'model': 'qwen3.5', 'created_at': '2026-07-02T02:58:15.4788103Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7165675200, 'load_duration': 5166967400, 'prompt_eval_count': 1701, 'prompt_eval_duration': 676908000, 'eval_count': 94, 'eval_duration': 1307868000, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'} id='lc_run--019f20c3-2738-7be3-a7f4-5538f6fbac41-0' tool_calls=[{'name': 'get_hero_best_positions', 'args': {'hero_name': 'BaJie'}, 'id': '938c2f07-10a8-4bad-b53a-b3740ddb10e2', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 1701, 'output_tokens': 94, 'total_tokens': 1795}

<class 'langchain_core.messages.tool.ToolMessage'>
content='{"hero_name

In [ ]:
model = ChatOllama(model="gemma4:12b")
prompt_class = model.invoke([
    {"system": "You are a query router.  Your job is to classify prompts for furher processing.  You must respond in this format:"
    "intent": "hero_info|draft_question|",
    "confident": "low|medium|high"}
])

print(res.content)


My name is Harry!


In [15]:
import heapq


def maxSum(nums: list[int], k: int, mul: int) -> int:
    max_sum = 0
    heapq.heapify_max(nums)
    for i in range(k):
        max = heapq.heappop_max(nums)

        if mul > 0:
            max_sum += max * mul
        else:
            max_sum += max

        mul -= 1

    return max_sum
    


print(maxSum(nums=[6,1,2,9], k=3, mul=2))
print(maxSum(nums=[3,7,5,2], k=2, mul=4))
print(maxSum(nums=[4,4], k=1, mul=1))
print(maxSum(nums=[3,7,13], k=3, mul=1))

            

26
43
4
23
